# GELU kernel: exact and tanh-approximate

GELU, the activation in BERT and GPT-2's MLP blocks, weights each input by the probability that a standard normal falls below it:

$$\mathrm{GELU}(x) = x\,\Phi(x) = \tfrac12\, x\left(1 + \operatorname{erf}\!\left(\tfrac{x}{\sqrt 2}\right)\right)$$

GPT-2 was trained with a cheaper approximation, and checkpoints must be served with the same formula they were trained with:

$$\mathrm{GELU}_{\tanh}(x) = \tfrac12\, x\left(1 + \tanh\!\left(\sqrt{2/\pi}\,\bigl(x + 0.044715\,x^3\bigr)\right)\right)$$

Write `gelu(x, approximate)` for a 1-D `float32` array of length `n` (possibly 0):

- `approximate` is a Python `bool`. `False` → exact formula using `math.erf`; `True` → tanh formula using `math.tanh`.
- Return a `float32` array of shape `(n,)`.
- Pass `approximate` to the kernel as a scalar argument; every thread reads the same flag, so there is no divergence.
- 128 threads per block, $\lceil n/128 \rceil$ blocks; no launch for `n == 0`.
- Compared with relative tolerance $10^{-5}$, absolute $10^{-6}$.

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import math
import numpy as np
from numba import cuda

TPB = 128

def gelu(x, approximate):
    """x: float32 (n,), approximate: bool. Returns GELU(x) as float32 (n,)."""
    n = x.shape[0]
    if n == 0:
        return np.zeros(0, dtype=np.float32)
    d_x = cuda.to_device(x)
    d_out = cuda.device_array(n, dtype=np.float32)
    gelu_kernel[(n + TPB - 1) // TPB, TPB](d_x, approximate, d_out)
    return d_out.copy_to_host()

@cuda.jit
def gelu_kernel(x, approximate, out):
    i = cuda.grid(1)
    if i < x.shape[0]:
        v = x[i]
        # TODO: exact (erf) or tanh-approximate GELU depending on the flag
        out[i] = v


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [-1.0, 0.0, 1.0, 2.0], "dtype": "float32"}, false], "e": {"__nd__": [-0.15865525603294373, 0.0, 0.8413447141647339, 1.9544997215270996], "dtype": "float32"}}, {"a": [{"__nd__": [-1.0, 0.0, 1.0, 2.0], "dtype": "float32"}, true], "e": {"__nd__": [-0.15880799293518066, 0.0, 0.8411920070648193, 1.9545977115631104], "dtype": "float32"}}, {"a": [{"__nd__": [], "dtype": "float32"}, true], "e": {"__nd__": [], "dtype": "float32"}}, {"a": [{"__nd__": [-30.0, -6.0, 6.0, 30.0], "dtype": "float32"}, false], "e": {"__nd__": [-0.0, -5.919535528420283e-09, 6.0, 30.0], "dtype": "float32"}}, {"a": [{"__nd__": [-30.0, -6.0, 6.0, 30.0], "dtype": "float32"}, true], "e": {"__nd__": [-0.0, -8.43964897967453e-11, 6.0, 30.0], "dtype": "float32"}}, {"a": [{"__nd__": [0.0036904599983245134, 0.8962365984916687, -0.8224135637283325, -2.6717755794525146, -1.364012360572815, -2.9749395847320557, 0.1804308146238327, 4.020645618438721, -1.4766196012496948, -1.8614246845245361, 1.4695261716842651, 1.0706610679626465, 0.3162427544593811, -2.7914042472839355, -0.0877554640173912, 2.085909605026245, -4.032643795013428, -1.372847318649292, -5.703668117523193, -3.8686132431030273, -5.525205135345459, -0.7052733898162842, -3.802339553833008, 0.8137930631637573, 0.47025325894355774, -0.560792863368988, -7.550279140472412, -1.6160787343978882, -0.14550283551216125, 0.33992695808410645, -4.590407371520996, -1.4332598447799683, -2.9355571269989014, -2.426511764526367, 3.1826958656311035, -2.4226040840148926, -0.09756511449813843, 2.653169631958008, -1.7508013248443604, -0.335105836391449, 0.33139243721961975, 0.19134531915187836, -3.6751675605773926, 0.228420689702034, 4.076470375061035, -4.641434192657471, 2.578148126602173, 0.35806208848953247, -1.9244111776351929, 6.00124979019165, 2.2867791652679443, -3.5978667736053467, 0.22354868054389954, 1.7300688028335571, -0.5663463473320007, 2.0487308502197266, -0.19955195486545563, 2.0017426013946533, 4.315567970275879, -2.026986837387085, 0.6094158291816711, -1.3899227380752563, 0.3818052411079407, -3.5615835189819336, -1.7379047870635986, -0.5885879397392273, 2.69629168510437, 3.435666084289551, -3.970583438873291, -2.3839271068573, 1.9407103061676025, -5.977259159088135, -1.3895095586776733, -0.29186078906059265, 3.771044969558716, 2.068211793899536, -0.9816402792930603, -1.1057276725769043, -0.750586211681366, 4.570588111877441, -1.2840747833251953, -0.9110411405563354, 1.0577671527862549, -0.36231133341789246, -0.5918526649475098, -3.3422014713287354, -0.03456440567970276, -1.330743670463562, 3.4983832836151123, 1.9592654705047607, -0.07243084162473679, 2.005143165588379, -1.0196086168289185, 3.156378984451294, -0.01619868166744709, 1.7501471042633057, -3.8726797103881836, 1.040040135383606, -5.06461238861084, -6.105987071990967, -0.9134306311607361, -2.6997828483581543, 0.4921583831310272, 6.734270095825195, -2.4951696395874023, -1.871830701828003, 0.6162118315696716, 1.4790399074554443, -0.5292181968688965, -0.6177909970283508, 2.107388973236084, 1.559722900390625, -3.101027488708496, -0.2375439554452896, 0.10586054623126984, -3.1634538173675537, 0.7795172929763794, -2.573869466781616, 2.9162001609802246, 0.5782377123832703, 0.2679194509983063, -1.773085117340088, -0.35582947731018066, -5.993238925933838, -3.3942224979400635, 1.0885194540023804, -6.3857011795043945, 2.539825677871704, -5.2382893562316895, 2.2702155113220215, -2.5364911556243896, 2.336973190307617, 0.39285361766815186, -4.610504627227783, 3.747446298599243, 4.3251214027404785, -0.19741471111774445, -0.8217487931251526, -0.4796009063720703, -2.925457000732422, 3.2957603931427, -1.6286758184432983, -0.1535712331533432, -2.3798892498016357, -1.8782192468643188, -3.8331754207611084, 3.7712080478668213, -0.46226271986961365, 2.8977649211883545, 0.03997379168868065, -2.0832104682922363, -0.9800558090209961, -1.6806931495666504, 0.023877298459410667, -1.1258004903793335, -0.899765133857727, -4.135724067687988, -2.4205377101898193, 4.962172508239746, -2.013699531555176, -3.1622812747955322, 1.0119789838790894, 4.221816539764404, -4.362072944641113, -0.6255655288696289, -1.8961576223373413, -5.2830586433410645, 2.204780101776123, -0.07033175230026245, 0.21432553231716156, -2.256934404373169, 1.3643524646759033, -1.6178920269012451, -0.42870962619781494, -3.324782371520996, -3.648308277130127, 4.006595611572266, -1.5213141441345215, 0.8750410676002502, -0.10137130320072174, -1.323435664176941, -1.5238828659057617, 1.8902478218078613, -0.9056028127670288, -0.4543309509754181, 0.0666646733880043, 3.5295250415802, 2.0415329933166504, 1.1478008031845093, -1.6907141208648682, -4.1459059715271, 2.8485898971557617, 2.899341344833374, -0.4221251904964447, 1.625651478767395, 2.3443291187286377, 2.493553400039673, 2.764150381088257, -1.3668529987335205, 4.544919013977051], "dtype": "float32"}, false], "e": {"__nd__": [0.0018506634514778852, 0.730376124382019, -0.16894082725048065, -0.010079420171678066, -0.11768949031829834, -0.004358973354101181, 0.1031329557299614, 4.020528793334961, -0.10319920629262924, -0.05834098532795906, 1.36541748046875, 0.9184548854827881, 0.1973642110824585, -0.00732463039457798, -0.04080940783023834, 2.0473339557647705, -0.00011120586714241654, -0.1165546253323555, -3.343951249235033e-08, -0.00021172183915041387, -9.091753838674776e-08, -0.16949133574962616, -0.0002725061494857073, 0.6446203589439392, 0.3202015459537506, -0.1612107753753662, -1.6387775468237886e-13, -0.08571462333202362, -0.06433508545160294, 0.21518880128860474, -1.0153573384741321e-05, -0.1087726578116417, -0.004886960610747337, -0.018495790660381317, 3.1803739070892334, -0.018665814772248268, -0.044991061091423035, 2.6425914764404297, -0.07001466304063797, -0.12357785552740097, 0.20871955156326294, 0.1101905032992363, -0.0004367817018646747, 0.13484595715999603, 4.076376914978027, -8.029664058994967e-06, 2.5653436183929443, 0.2291065901517868, -0.05225072056055069, 6.00124979019165, 2.261385917663574, -0.0005771660362370312, 0.1315462589263916, 1.6577363014221191, -0.16173672676086426, 2.007255792617798, -0.08399451524019241, 1.9563907384872437, 4.315533638000488, -0.043239448219537735, 0.44418832659721375, -0.11435752362012863, 0.24767594039440155, -0.0006564444629475474, -0.07145187258720398, -0.1636679768562317, 2.6868388652801514, 3.4346506595611572, -0.00014233913680072874, -0.020417138934135437, 1.8899670839309692, -6.78188305514027e-09, -0.11441074311733246, -0.11242376267910004, 3.770738363265991, 2.0282745361328125, -0.1601433902978897, -0.14863435924053192, -0.16997089982032776, 4.570577144622803, -0.12783977389335632, -0.16502311825752258, 0.9043054580688477, -0.12991024553775787, -0.16392818093299866, -0.0013889650581404567, -0.016805682331323624, -0.12194494158029556, 3.4975643157958984, 1.9102038145065308, -0.034124307334423065, 1.9600797891616821, -0.15697593986988068, 3.153857946395874, -0.007994663901627064, 1.6800599098205566, -0.0002084377920255065, 0.8849071264266968, -1.0363006595071056e-06, -3.1191569327404522e-09, -0.16488158702850342, -0.009366189129650593, 0.33894768357276917, 6.734270095825195, -0.015706710517406464, -0.05730612203478813, 0.4505264163017273, 1.3761507272720337, -0.15788011252880096, -0.16578826308250427, 2.0704212188720703, 1.4670555591583252, -0.002990171080455184, -0.09647073596715927, 0.0573926642537117, -0.0024660564959049225, 0.6097092032432556, -0.012942473404109478, 2.9110336303710938, 0.4154338240623474, 0.16225719451904297, -0.06756750494241714, -0.12844879925251007, -6.1641229898157235e-09, -0.0011680151801556349, 0.9381046295166016, -5.447438611305699e-10, 2.5257413387298584, -4.244899400873692e-07, 2.243887186050415, -0.01420049462467432, 2.3142571449279785, 0.256449431180954, -9.260037586500403e-06, 3.7471115589141846, 4.3250885009765625, -0.0832599475979805, -0.16895973682403564, -0.1514366716146469, -0.005031066481024027, 3.294142961502075, -0.08418761193752289, -0.06741376221179962, -0.020607274025678635, -0.056676384061574936, -0.00024244784435722977, 3.770901679992676, -0.14882385730743408, 2.892319679260254, 0.020624198019504547, -0.03878108784556389, -0.16026782989501953, -0.07800310850143433, 0.012166074477136135, -0.1464947760105133, -0.16566713154315948, -7.316869596252218e-05, -0.018756229430437088, 4.962170600891113, -0.044342756271362305, -0.002475091954693198, 0.8543388843536377, 4.221765518188477, -2.8099608243792318e-05, -0.166275292634964, -0.05493094399571419, -3.3559246048753266e-07, 2.1744980812072754, -0.03319411352276802, 0.1253490149974823, -0.027096951380372047, 1.2467066049575806, -0.08549417555332184, -0.14321786165237427, -0.0014710092218592763, -0.0004815274151042104, 4.006472110748291, -0.09750181436538696, 0.7081044316291809, -0.046593077480793, -0.12287473678588867, -0.09717646986246109, 1.8347455263137817, -0.1653386950492859, -0.1475645750761032, 0.035103995352983475, 3.5287904739379883, 1.9994795322418213, 1.0037226676940918, -0.07683570683002472, -7.016382005531341e-05, 2.8423354625701904, 2.893920421600342, -0.14203111827373505, 1.5410982370376587, 2.321986198425293, 2.477785110473633, 2.75626277923584, -0.11732477694749832, 4.5449066162109375], "dtype": "float32"}}, {"a": [{"__nd__": [0.0036904599983245134, 0.8962365984916687, -0.8224135637283325, -2.6717755794525146, -1.364012360572815, -2.9749395847320557, 0.1804308146238327, 4.020645618438721, -1.4766196012496948, -1.8614246845245361, 1.4695261716842651, 1.0706610679626465, 0.3162427544593811, -2.7914042472839355, -0.0877554640173912, 2.085909605026245, -4.032643795013428, -1.372847318649292, -5.703668117523193, -3.8686132431030273, -5.525205135345459, -0.7052733898162842, -3.802339553833008, 0.8137930631637573, 0.47025325894355774, -0.560792863368988, -7.550279140472412, -1.6160787343978882, -0.14550283551216125, 0.33992695808410645, -4.590407371520996, -1.4332598447799683, -2.9355571269989014, -2.426511764526367, 3.1826958656311035, -2.4226040840148926, -0.09756511449813843, 2.653169631958008, -1.7508013248443604, -0.335105836391449, 0.33139243721961975, 0.19134531915187836, -3.6751675605773926, 0.228420689702034, 4.076470375061035, -4.641434192657471, 2.578148126602173, 0.35806208848953247, -1.9244111776351929, 6.00124979019165, 2.2867791652679443, -3.5978667736053467, 0.22354868054389954, 1.7300688028335571, -0.5663463473320007, 2.0487308502197266, -0.19955195486545563, 2.0017426013946533, 4.315567970275879, -2.026986837387085, 0.6094158291816711, -1.3899227380752563, 0.3818052411079407, -3.5615835189819336, -1.7379047870635986, -0.5885879397392273, 2.69629168510437, 3.435666084289551, -3.970583438873291, -2.3839271068573, 1.9407103061676025, -5.977259159088135, -1.3895095586776733, -0.29186078906059265, 3.771044969558716, 2.068211793899536, -0.9816402792930603, -1.1057276725769043, -0.750586211681366, 4.570588111877441, -1.2840747833251953, -0.9110411405563354, 1.0577671527862549, -0.36231133341789246, -0.5918526649475098, -3.3422014713287354, -0.03456440567970276, -1.330743670463562, 3.4983832836151123, 1.9592654705047607, -0.07243084162473679, 2.005143165588379, -1.0196086168289185, 3.156378984451294, -0.01619868166744709, 1.7501471042633057, -3.8726797103881836, 1.040040135383606, -5.06461238861084, -6.105987071990967, -0.9134306311607361, -2.6997828483581543, 0.4921583831310272, 6.734270095825195, -2.4951696395874023, -1.871830701828003, 0.6162118315696716, 1.4790399074554443, -0.5292181968688965, -0.6177909970283508, 2.107388973236084, 1.559722900390625, -3.101027488708496, -0.2375439554452896, 0.10586054623126984, -3.1634538173675537, 0.7795172929763794, -2.573869466781616, 2.9162001609802246, 0.5782377123832703, 0.2679194509983063, -1.773085117340088, -0.35582947731018066, -5.993238925933838, -3.3942224979400635, 1.0885194540023804, -6.3857011795043945, 2.539825677871704, -5.2382893562316895, 2.2702155113220215, -2.5364911556243896, 2.336973190307617, 0.39285361766815186, -4.610504627227783, 3.747446298599243, 4.3251214027404785, -0.19741471111774445, -0.8217487931251526, -0.4796009063720703, -2.925457000732422, 3.2957603931427, -1.6286758184432983, -0.1535712331533432, -2.3798892498016357, -1.8782192468643188, -3.8331754207611084, 3.7712080478668213, -0.46226271986961365, 2.8977649211883545, 0.03997379168868065, -2.0832104682922363, -0.9800558090209961, -1.6806931495666504, 0.023877298459410667, -1.1258004903793335, -0.899765133857727, -4.135724067687988, -2.4205377101898193, 4.962172508239746, -2.013699531555176, -3.1622812747955322, 1.0119789838790894, 4.221816539764404, -4.362072944641113, -0.6255655288696289, -1.8961576223373413, -5.2830586433410645, 2.204780101776123, -0.07033175230026245, 0.21432553231716156, -2.256934404373169, 1.3643524646759033, -1.6178920269012451, -0.42870962619781494, -3.324782371520996, -3.648308277130127, 4.006595611572266, -1.5213141441345215, 0.8750410676002502, -0.10137130320072174, -1.323435664176941, -1.5238828659057617, 1.8902478218078613, -0.9056028127670288, -0.4543309509754181, 0.0666646733880043, 3.5295250415802, 2.0415329933166504, 1.1478008031845093, -1.6907141208648682, -4.1459059715271, 2.8485898971557617, 2.899341344833374, -0.4221251904964447, 1.625651478767395, 2.3443291187286377, 2.493553400039673, 2.764150381088257, -1.3668529987335205, 4.544919013977051], "dtype": "float32"}, true], "e": {"__nd__": [0.0018506634514778852, 0.7302601337432861, -0.16903197765350342, -0.009606779552996159, -0.11792316287755966, -0.003937857691198587, 0.10313261300325394, 4.0205817222595215, -0.10342195630073547, -0.05835968255996704, 1.3651933670043945, 0.9182780385017395, 0.19736114144325256, -0.006857918109744787, -0.040809426456689835, 2.047506332397461, -6.005852992529981e-05, -0.11678822338581085, -1.130541549265729e-09, -0.000129494204884395, -4.855027047057092e-09, -0.16954798996448517, -0.00017430503794457763, 0.6445319652557373, 0.3201877772808075, -0.16123676300048828, -0.0, -0.08589170128107071, -0.06433523446321487, 0.21518474817276, -3.041090621991316e-06, -0.10900231450796127, -0.004453051369637251, -0.018085595220327377, 3.180711030960083, -0.018257450312376022, -0.04499109089374542, 2.6430630683898926, -0.07011522352695465, -0.12358169257640839, 0.20871588587760925, 0.11019007116556168, -0.00030194874852895737, 0.1348450928926468, 4.076421737670898, -2.2459032607002882e-06, 2.5658047199249268, 0.22910165786743164, -0.052217595279216766, 6.00124979019165, 2.2617156505584717, -0.00041619548574090004, 0.13154548406600952, 1.657622218132019, -0.16176362335681915, 2.0073962211608887, -0.083995021879673, 1.9564902782440186, 4.315553665161133, -0.04311804100871086, 0.4441535770893097, -0.11459064483642578, 0.24766963720321655, -0.0004822338523808867, -0.07156097143888474, -0.16369879245758057, 2.687312364578247, 3.434875249862671, -8.076740778051317e-05, -0.020028309896588326, 1.8900140523910522, -1.0381400600278923e-10, -0.1146438717842102, -0.11242601275444031, 3.7708449363708496, 2.0284316539764404, -0.16028964519500732, -0.14882241189479828, -0.17003999650478363, 4.570584774017334, -0.12806835770606995, -0.16514438390731812, 0.9041328430175781, -0.12991541624069214, -0.1639595776796341, -0.0011238346341997385, -0.016805682331323624, -0.12217767536640167, 3.497763156890869, 1.910266637802124, -0.03412431851029396, 1.9601823091506958, -0.15713554620742798, 3.1542065143585205, -0.007994663901627064, 1.6799589395523071, -0.0001271231158170849, 0.8847404718399048, -1.47520836435433e-07, -3.158305561523633e-11, -0.1650036871433258, -0.008892949670553207, 0.3389314115047455, 6.734270095825195, -0.015268474817276001, -0.05731646716594696, 0.4504903256893158, 1.3759284019470215, -0.15790124237537384, -0.16582466661930084, 2.0706117153167725, 1.466855764389038, -0.0026176448445767164, -0.09647174179553986, 0.057392627000808716, -0.0021204985678195953, 0.6096314787864685, -0.012482216581702232, 2.911473512649536, 0.41540491580963135, 0.16225560009479523, -0.06765271723270416, -0.12845362722873688, -8.976817900130385e-11, -0.0009258014615625143, 0.9379220008850098, -2.0439179862496992e-12, 2.526193380355835, -4.311348078545052e-08, 2.2442054748535156, -0.013749330304563046, 2.3146190643310547, 0.2564423978328705, -2.700571940295049e-06, 3.7472245693206787, 4.325108051300049, -0.0832604393362999, -0.1690506488084793, -0.1514514684677124, -0.004594108089804649, 3.294428825378418, -0.084358811378479, -0.06741394847631454, -0.020220624282956123, -0.05668153241276741, -0.0001519381912657991, 3.771008253097534, -0.14883677661418915, 2.8927643299102783, 0.020624198019504547, -0.038611117750406265, -0.16041354835033417, -0.07814712822437286, 0.012166074477136135, -0.14668887853622437, -0.16578443348407745, -3.6161123716738075e-05, -0.018348846584558487, 4.962172031402588, -0.04423290491104126, -0.0021290145814418793, 0.8541819453239441, 4.221793174743652, -1.1079931937274523e-05, -0.1663133054971695, -0.05492141842842102, -3.105263246538925e-08, 2.1747677326202393, -0.03319412097334862, 0.12534834444522858, -0.026788219809532166, 1.246472954750061, -0.08567039668560028, -0.14322762191295624, -0.0011980910785496235, -0.0003379394765943289, 4.006527423858643, -0.09771384298801422, 0.7079957127571106, -0.04659310728311539, -0.12310705333948135, -0.09738779067993164, 1.8347502946853638, -0.16545803844928741, -0.14757667481899261, 0.03510398790240288, 3.5289769172668457, 1.9996135234832764, 1.003522276878357, -0.07697398215532303, -3.4357613913016394e-05, 2.84279203414917, 2.894364595413208, -0.14204032719135284, 1.5409256219863892, 2.322352409362793, 2.4782228469848633, 2.756732702255249, -0.11755843460559845, 4.544914722442627], "dtype": "float32"}}, {"a": [{"__nd__": [-4.0, -3.9375, -3.875, -3.8125, -3.75, -3.6875, -3.625, -3.5625, -3.5, -3.4375, -3.375, -3.3125, -3.25, -3.1875, -3.125, -3.0625, -3.0, -2.9375, -2.875, -2.8125, -2.75, -2.6875, -2.625, -2.5625, -2.5, -2.4375, -2.375, -2.3125, -2.25, -2.1875, -2.125, -2.0625, -2.0, -1.9375, -1.875, -1.8125, -1.75, -1.6875, -1.625, -1.5625, -1.5, -1.4375, -1.375, -1.3125, -1.25, -1.1875, -1.125, -1.0625, -1.0, -0.9375, -0.875, -0.8125, -0.75, -0.6875, -0.625, -0.5625, -0.5, -0.4375, -0.375, -0.3125, -0.25, -0.1875, -0.125, -0.0625, 0.0, 0.0625, 0.125, 0.1875, 0.25, 0.3125, 0.375, 0.4375, 0.5, 0.5625, 0.625, 0.6875, 0.75, 0.8125, 0.875, 0.9375, 1.0, 1.0625, 1.125, 1.1875, 1.25, 1.3125, 1.375, 1.4375, 1.5, 1.5625, 1.625, 1.6875, 1.75, 1.8125, 1.875, 1.9375, 2.0, 2.0625, 2.125, 2.1875, 2.25, 2.3125, 2.375, 2.4375, 2.5, 2.5625, 2.625, 2.6875, 2.75, 2.8125, 2.875, 2.9375, 3.0, 3.0625, 3.125, 3.1875, 3.25, 3.3125, 3.375, 3.4375, 3.5, 3.5625, 3.625, 3.6875, 3.75, 3.8125, 3.875, 3.9375, 4.0], "dtype": "float32"}, true], "e": {"__nd__": [-7.024591468507424e-05, -9.432250953977928e-05, -0.00012578816676978022, -0.00016662316920701414, -0.00021925214969087392, -0.0002866192371584475, -0.00037227332359179854, -0.00048045566654764116, -0.0006161972414702177, -0.0007854183204472065, -0.0009950286475941539, -0.001253032125532627, -0.001568627543747425, -0.0019522927468642592, -0.002415874507278204, -0.0029726538341492414, -0.0036373911425471306, -0.0044263447634875774, -0.005357267800718546, -0.006449350621551275, -0.0077231451869010925, -0.009200405329465866, -0.010903908871114254, -0.012857201509177685, -0.015084263868629932, -0.0176091268658638, -0.020455406978726387, -0.02364577353000641, -0.0272013321518898, -0.031140921637415886, -0.03548039123415947, -0.04023176059126854, -0.04540230706334114, -0.050993677228689194, -0.057000964879989624, -0.06341162323951721, -0.07020461559295654, -0.07734938710927963, -0.08480502665042877, -0.0925193801522255, -0.10042841732501984, -0.1084555983543396, -0.11651143431663513, -0.12449340522289276, -0.13228578865528107, -0.13976001739501953, -0.14677520096302032, -0.1531788557767868, -0.15880799293518066, -0.1634906381368637, -0.16704732179641724, -0.16929301619529724, -0.17003944516181946, -0.16909728944301605, -0.16627885401248932, -0.16140060126781464, -0.1542859822511673, -0.14476807415485382, -0.13269223272800446, -0.11791864037513733, -0.10032464563846588, -0.07980683445930481, -0.05628279969096184, -0.029692649841308594, 0.0, 0.032807350158691406, 0.06871719658374786, 0.10769317299127579, 0.14967535436153412, 0.19458135962486267, 0.24230776727199554, 0.292731910943985, 0.3457140028476715, 0.40109938383102417, 0.4587211608886719, 0.5184026956558228, 0.5799605846405029, 0.6432070136070251, 0.7079526782035828, 0.7740093469619751, 0.8411920070648193, 0.9093211889266968, 0.9782247543334961, 1.0477399826049805, 1.1177141666412354, 1.1880065202713013, 1.2584885358810425, 1.3290443420410156, 1.3995715379714966, 1.4699805974960327, 1.5401948690414429, 1.6101505756378174, 1.6797953844070435, 1.7490884065628052, 1.817999005317688, 1.8865063190460205, 1.9545977115631104, 2.022268295288086, 2.089519739151001, 2.1563589572906494, 2.2227985858917236, 2.2888543605804443, 2.3545446395874023, 2.419890880584717, 2.4849157333374023, 2.54964280128479, 2.614096164703369, 2.67829966545105, 2.742276906967163, 2.8060505390167236, 2.869642734527588, 2.9330735206604004, 2.9963626861572266, 3.0595273971557617, 3.122584104537964, 3.1855475902557373, 3.2484312057495117, 3.3112471103668213, 3.374005079269409, 3.4367146492004395, 3.4993836879730225, 3.5620195865631104, 3.6246278285980225, 3.68721342086792, 3.7497806549072266, 3.812333345413208, 3.8748741149902344, 3.937405586242676, 3.999929666519165], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(gelu, json.loads(_TESTS), 'cuda', *[1e-05, 1e-06])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")